# 🤟 Wasel v4 Pro — Gemini Live Sign Language Translator
### Powered by Google Gemini AI — No custom model needed!

**How it works:**
- 📸 Webcam captures your sign gesture
- 🧠 Gemini AI interprets the image instantly
- 📝 Translation appears on screen!

**Steps:** Cell 1 (Setup) → Cell 2 (Launch!)

> ⚡ **Zero training, zero MediaPipe, zero YOLO — pure Gemini AI!**

In [ ]:
#@title 📦 Cell 1: Setup

!pip install -q "google-genai>=1.0.0" "gradio>=5.0.0" "opencv-python-headless" "Pillow"

import google.genai as genai
print(f"✅ google-genai ready")
import gradio as gr
print(f"✅ gradio: {gr.__version__}")
print("\n🎉 Proceed to Cell 2!")

In [ ]:
#@title 🎥 Cell 2: Launch Live Translator
#@markdown Get a FREE Gemini API key from: https://aistudio.google.com/apikey

GEMINI_API_KEY = "" #@param {type:"string"}

import cv2, time, io, base64, threading
import numpy as np
import gradio as gr
from PIL import Image
import google.genai as genai
from google.genai import types

if not GEMINI_API_KEY:
    raise ValueError("❌ Paste your Gemini API key! Get one free at: https://aistudio.google.com/apikey")

# ─── Setup Gemini ───
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.0-flash"

# Test connection
test = client.models.generate_content(
    model=MODEL,
    contents="Say 'Wasel v4 connected!' in 3 words max"
)
print(f"✅ Gemini connected: {test.text.strip()}")

# ─── Sign Language Prompt ───
SYSTEM_PROMPT = """You are a sign language interpreter AI. 
Analyze the image and identify any sign language gesture being performed.

Rules:
1. If you see a clear hand gesture or sign, respond with ONLY the word/letter being signed.
   Format: SIGN: [word] (CONFIDENCE: high/medium/low)
2. If no clear sign gesture is visible, respond: WAITING
3. Look for hand shapes, finger positions, and body posture.
4. Consider ASL, PSL (Pakistan), and common international signs.
5. Keep response under 10 words.
"""

# ─── Frame Processing ───
last_translation = "Waiting for signs..."
last_call_time = 0
MIN_INTERVAL = 1.5  # seconds between API calls (rate limiting)
lock = threading.Lock()

def frame_to_pil(frame):
    """Convert numpy frame to PIL Image."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)

def call_gemini(pil_image):
    """Send image to Gemini and get translation."""
    global last_translation, last_call_time
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=[
                SYSTEM_PROMPT,
                pil_image
            ],
            config=types.GenerateContentConfig(
                max_output_tokens=50,
                temperature=0.1,
            )
        )
        text = response.text.strip()
        with lock:
            last_translation = text
            last_call_time = time.time()
    except Exception as e:
        with lock:
            last_translation = f"Error: {str(e)[:50]}"

def translate(image):
    """Process each webcam frame."""
    global last_call_time
    if image is None:
        return image

    out = image.copy()
    now = time.time()

    # Call Gemini every MIN_INTERVAL seconds (async)
    if now - last_call_time >= MIN_INTERVAL:
        with lock:
            last_call_time = now  # prevent duplicate calls
        pil = frame_to_pil(image)
        # Resize for faster API call
        pil.thumbnail((640, 480))
        t = threading.Thread(target=call_gemini, args=(pil,), daemon=True)
        t.start()

    # Draw translation on frame
    with lock:
        text = last_translation

    # Background box
    h, w = out.shape[:2]
    cv2.rectangle(out, (0, 0), (w, 70), (0, 0, 0), -1)

    # Color based on status
    if text.startswith("SIGN:"):
        color = (0, 255, 0)  # Green = detected
        display = text.replace("SIGN: ", "🤟 ")
    elif text == "WAITING":
        color = (255, 255, 0)  # Yellow = waiting
        display = "⏳ Show a sign..."
    elif text.startswith("Error"):
        color = (0, 0, 255)  # Red = error
        display = text
    else:
        color = (0, 200, 255)  # Orange = other
        display = text

    cv2.putText(out, display, (15, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)

    return out

# ─── Launch ───
demo = gr.Interface(
    fn=translate,
    inputs=gr.Image(sources=["webcam"], streaming=True, label="📸 Show Your Signs"),
    outputs=gr.Image(label="🤟 Gemini Translation"),
    live=True,
    title="🤟 Wasel v4 Pro — Gemini AI Sign Language Translator",
    description="Powered by Google Gemini AI. Show sign language gestures and get instant translation!",
)

print("\n🎉 Launching Wasel v4 with Gemini AI...")
print("📋 A public URL will appear below!")
print(f"⚡ Using model: {MODEL}")
print(f"⏱️ API call interval: {MIN_INTERVAL}s\n")
demo.launch(share=True, quiet=False)